# Kest Lineage Tracking & Integrity Verification

This notebook demonstrates how to use Kest to track data lineage, visualize the resulting DAG as an ASCII tree, and finally prove that tampering with historical outputs immediately invalidates the cryptographic signature.

In [ ]:
from kest import config, render_passport
from kest.core.crypto import generate_keypair, sign_passport
from kest.core.policy import LocalOpaEngine
from kest.presentation.decorators import kest_verified as verified

# 1. Setup Kest Environment & Keys
priv_key, pub_key = generate_keypair()
# 2. Configure Verification Key to actively reject bad actors at ingress
config.verification_key = pub_key
# 3. Setup OPA engine for policy enforcement
try:
    config.policy_engine = LocalOpaEngine()
    policy = """
package kest.policy
default allow = false
allow {
    not unsafe_mix
}
unsafe_mix {
    has_pii
    not has_stripped
}
has_pii { input.taints[_] == "pii_data" }
has_stripped { input.taints[_] == "pii_stripped" }
"""
    config.policy_engine.add_policy("access", policy)
except Exception as e:
    from kest.core.policy import OpaEngine

    class MockWorkaroundEngine(OpaEngine):
        def evaluate(self, payload: dict, rule_path: str) -> bool:
            print(f"   [MockEngine] Evaluating rule: {rule_path}")
            if rule_path.endswith("allow"):
                taints = payload.get("taints", [])
                has_pii = "pii_data" in taints
                has_stripped = "pii_stripped" in taints
                return not (has_pii and not has_stripped)
            return False

    print(f"Warning: LocalOpaEngine unavailable ({e}). Using MockWorkaroundEngine.")
    config.policy_engine = MockWorkaroundEngine()


# 4. Define Tainted Functions
@verified(added_taint=["pii_data"])
def ingest_pii(data: dict):
    return data


@verified(added_taint=["internet_data"])
def fetch_internet_data(query: str):
    return {"source": "internet", "query": query}


@verified(added_taint=["pii_stripped"])
def clean_pii(data: dict):
    clean = data.copy()
    if "ssn" in clean:
        clean["ssn"] = "***-**-****"
    return clean


@verified(enforce_rules=["data.kest.policy.allow"])
def merge_results(a, b):
    return {"merged": True, "a": a, "b": b}

### Demonstrate Policy Failure

Our OPA policy enforces that `merge_results` cannot accept `pii_data` unless it also has the `pii_stripped` taint. Let's try merging unstripped data.

In [ ]:
try:
    raw_pii = ingest_pii({"user": "Bob", "ssn": "999-99-999"})
    web_data = fetch_internet_data("news")
    # This should trigger a PermissionError because the PII hasn't been stripped
    merge_results(raw_pii, web_data)
    print("❌ FAIL: Executed when it shouldn't have!")
except PermissionError as e:
    print(f"✅ POLICY ENFORCED! Execution blocked: {e}")

### Execute the Pipeline

We will run a flow that ingests PII, cleans it, fetches internet data, and then merges them.

In [ ]:
# Ingest and process
raw_pii = ingest_pii({"user": "Alice", "ssn": "123-45-678"})
cleaned_pii = clean_pii(raw_pii)

# Fetch internet data
web_data = fetch_internet_data("weather")

# Merge everything
final_result = merge_results(cleaned_pii, web_data)

print("Final Data:", final_result.data)
print("Lineage Nodes:", len(final_result.passport.history))

### Visualize the DAG (ASCII Tree)

We use `render_passport` to print the recursive graph history, showing node merges.

In [ ]:
print("---- DAG LINEAGE ----")
ascii_tree = render_passport(final_result.passport.model_dump())

# Sign the final passport so we can simulate tampering
jws_token = sign_passport(priv_key, final_result.passport, kid="demo-key-1")
print(f"\nPassport Signed. JWS Token: {jws_token[:50]}...")

### Simulate Malicious Tampering & Active Rejection

What happens if an attacker intercepts the result and historically alters the passport to remove the `pii_data` taint from the ingest node? \n
If we configure Kest with a `verification_key`, any `@verified` boundary function will subsequently enforce the ED25519 signature before allowing execution.


In [ ]:
import jwt

from kest.core.crypto import verify_signature

# 1. Provide the tampered token to an ingress
print("Simulating attacker altering the JWS payload to drop 'pii_data'...")
decoded_payload = jwt.decode(jws_token, options={"verify_signature": False})

for entry_id, entry in decoded_payload["history"].items():
    if entry["node_id"] == "ingest_pii":
        entry["accumulated_taint"] = []
        entry["added_taint"] = []

# Attacker re-encodes the maliciously altered payload with a fake signature/algorithm
tampered_jws = jwt.encode(
    decoded_payload, "fake_attacker_key_for_demonstration_purposes", algorithm="HS256"
)

# 2. Attempt to process the tampered data in a secure boundary ingress
try:
    print("\nAttempting to verify tampered JWS on network ingress...")
    verify_signature(pub_key, tampered_jws)
    print("\n❌ FAIL: Executed when it shouldn't have!")
except Exception as e:
    print(f"\n✅ INGRESS REJECTED. Tampering Detected! Error:\n{e}")